In [3]:
import pandas as pd 
import numpy as np 
import pdfplumber
import re 
import nltk
from nltk import sent_tokenize
from google import genai
from google.genai import types

from pathlib import Path
import faiss
from dotenv import load_dotenv
import os 


In [5]:
load_dotenv()

True

In [6]:
client = genai.Client(
    api_key= os.getenv('GEMINI_API_KEY')
)

In [25]:
pdf_data = '../data/ShaheerResume0.pdf'
projects = Path('../data/projects')

In [26]:
def pdf_upload():

    with pdfplumber.open(pdf_data) as pdf:
        file = pdf.pages[0]

        text = file.extract_text()
        print(text)
    

    
    return text


    

pdf_file = pdf_upload()

SHAHEER RANGREJ
Location: Mumbai, Maharashtra Email: whyshaheer143@gmail.com
LinkedIn:shaheer-rangrej GitHub:github.com/B-Samy
PROFESSIONAL SUMMARY
Aspiring Data Scientist and AI/ML Engineer with practical hands-on experience in Machine Learning, Natural
Language Processing (NLP), and Deep Learning. Proven track record of developing end-to-end intelligent systems
including AI resume parsers, fraud detection, sentiment analysis, and predictive analytics models. Passionate about
applying data-driven techniques to solve real-world problems.
TECHNICAL SKILLS
Core Languages: Python, SQL
AI / ML / Data Science: Machine Learning, Deep Learning, Natural Language Processing (NLP), Data Analysis,
Predictive Modeling
Soft Skills: Effective Communication, Problem-Solving, Analytical Thinking, Teamwork
PROJECTS
AI Resume Parser
Designed and implemented an NLP-powered resume parsing system capable of automatically extracting candidate
details, skills, and experience from unstructured text documents.

In [27]:

def file_handling():
    all_text = []

    for md_file in Path(projects).rglob("*.md"):
        text = md_file.read_text(encoding="utf-8")
        all_text.append(text)

    return "\n\n".join(all_text)

In [28]:
def main(pdf_file , file_handling):
    readme_text = file_handling()

    merged_text = f"""
RESUME : 

{pdf_file}

 README / PROJECT CONTENT : 

{readme_text}
"""

    return merged_text


text_merger = main(pdf_file , file_handling=file_handling)

print(text_merger)


RESUME : 

SHAHEER RANGREJ
Location: Mumbai, Maharashtra Email: whyshaheer143@gmail.com
LinkedIn:shaheer-rangrej GitHub:github.com/B-Samy
PROFESSIONAL SUMMARY
Aspiring Data Scientist and AI/ML Engineer with practical hands-on experience in Machine Learning, Natural
Language Processing (NLP), and Deep Learning. Proven track record of developing end-to-end intelligent systems
including AI resume parsers, fraud detection, sentiment analysis, and predictive analytics models. Passionate about
applying data-driven techniques to solve real-world problems.
TECHNICAL SKILLS
Core Languages: Python, SQL
AI / ML / Data Science: Machine Learning, Deep Learning, Natural Language Processing (NLP), Data Analysis,
Predictive Modeling
Soft Skills: Effective Communication, Problem-Solving, Analytical Thinking, Teamwork
PROJECTS
AI Resume Parser
Designed and implemented an NLP-powered resume parsing system capable of automatically extracting candidate
details, skills, and experience from unstructured tex

In [29]:
def preprocess(text):
    text = re.sub(r'\s+' , " " , text)

    return text



clean_text = preprocess(text_merger)
print(clean_text)

 RESUME : SHAHEER RANGREJ Location: Mumbai, Maharashtra Email: whyshaheer143@gmail.com LinkedIn:shaheer-rangrej GitHub:github.com/B-Samy PROFESSIONAL SUMMARY Aspiring Data Scientist and AI/ML Engineer with practical hands-on experience in Machine Learning, Natural Language Processing (NLP), and Deep Learning. Proven track record of developing end-to-end intelligent systems including AI resume parsers, fraud detection, sentiment analysis, and predictive analytics models. Passionate about applying data-driven techniques to solve real-world problems. TECHNICAL SKILLS Core Languages: Python, SQL AI / ML / Data Science: Machine Learning, Deep Learning, Natural Language Processing (NLP), Data Analysis, Predictive Modeling Soft Skills: Effective Communication, Problem-Solving, Analytical Thinking, Teamwork PROJECTS AI Resume Parser Designed and implemented an NLP-powered resume parsing system capable of automatically extracting candidate details, skills, and experience from unstructured text 

In [30]:
sen_chunk = sent_tokenize(clean_text)
print(sen_chunk)

[' RESUME : SHAHEER RANGREJ Location: Mumbai, Maharashtra Email: whyshaheer143@gmail.com LinkedIn:shaheer-rangrej GitHub:github.com/B-Samy PROFESSIONAL SUMMARY Aspiring Data Scientist and AI/ML Engineer with practical hands-on experience in Machine Learning, Natural Language Processing (NLP), and Deep Learning.', 'Proven track record of developing end-to-end intelligent systems including AI resume parsers, fraud detection, sentiment analysis, and predictive analytics models.', 'Passionate about applying data-driven techniques to solve real-world problems.', 'TECHNICAL SKILLS Core Languages: Python, SQL AI / ML / Data Science: Machine Learning, Deep Learning, Natural Language Processing (NLP), Data Analysis, Predictive Modeling Soft Skills: Effective Communication, Problem-Solving, Analytical Thinking, Teamwork PROJECTS AI Resume Parser Designed and implemented an NLP-powered resume parsing system capable of automatically extracting candidate details, skills, and experience from unstruc

In [31]:
chunks = []


chunk_size = 5


for i in range(0 , len(sen_chunk) , chunk_size):
    chunk = sen_chunk[i:i + chunk_size]

    chunks.append(" ".join(chunk))

print(chunks[0])

 RESUME : SHAHEER RANGREJ Location: Mumbai, Maharashtra Email: whyshaheer143@gmail.com LinkedIn:shaheer-rangrej GitHub:github.com/B-Samy PROFESSIONAL SUMMARY Aspiring Data Scientist and AI/ML Engineer with practical hands-on experience in Machine Learning, Natural Language Processing (NLP), and Deep Learning. Proven track record of developing end-to-end intelligent systems including AI resume parsers, fraud detection, sentiment analysis, and predictive analytics models. Passionate about applying data-driven techniques to solve real-world problems. TECHNICAL SKILLS Core Languages: Python, SQL AI / ML / Data Science: Machine Learning, Deep Learning, Natural Language Processing (NLP), Data Analysis, Predictive Modeling Soft Skills: Effective Communication, Problem-Solving, Analytical Thinking, Teamwork PROJECTS AI Resume Parser Designed and implemented an NLP-powered resume parsing system capable of automatically extracting candidate details, skills, and experience from unstructured text 

In [32]:
response = client.models.embed_content(
    model='gemini-embedding-001',
    contents=chunks,
)


doucment_embedding = np.array(
    [embedding.values for embedding in response.embeddings],
    dtype='float32'
)



In [33]:
dimension = doucment_embedding.shape[1]
print(dimension)

3072


In [34]:
index = faiss.IndexFlatL2(dimension)
index.add(doucment_embedding)


In [35]:
question = 'Which project convert lectures into notes ?'


query_response = client.models.embed_content(
    model='gemini-embedding-001',
    contents=question
)


query_embedding = np.array(
    [query_response.embeddings[0].values] ,
    dtype='float32'
)


distance , indices = index.search(
    query_embedding,
    k=5
)

print(indices)

[[ 8  3  7 10  9]]


In [36]:

print('Question : \n')



print(question)



Question : 

Which project convert lectures into notes ?


In [37]:

relevant_chunks = [
    chunks[i]
    for i in indices[0]
]

print(relevant_chunks)



['Upload lecture ↓ 2. Transcribe audio ↓ 3. Clean transcript ↓ 4. Sentence tokenize ↓ 5. Create chunks ↓ 6.', 'Audio Input The user provides a lecture audio file. EduVoice uses a **Speech-to-Text model** to convert the audio into text. ```text lecture.mp3 ↓ Speech-to-Text ↓ Raw transcript ``` --- ## 2. Text Processing The transcript is cleaned before further processing. The system removes unnecessary speech, filler words, repetition, and other irrelevant conversational content while preserving important technical information.', '--- ## 7. LLM Generation The retrieved content is provided to the **Gemini API** along with instructions. The LLM understands the lecture context and generates structured learning material. The prompt instructs the model to preserve important technical information, remove unnecessary speech, avoid hallucinating information, and generate content supported by the lecture. --- # 🏗️ Project Architecture ```text 🎙️ Lecture Audio │ ▼ ┌─────────────────┐ │ Speech-to-T

In [38]:
question = "Which project converts lectures into notes?"



In [41]:
def ask_question(question):

    query_response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=question
    )

    query_embedding = np.array(
        [query_response.embeddings[0].values],
        dtype="float32"
    )

    distance, indices = index.search(
        query_embedding,
        k=5
    )

    relevant_chunks = [
        chunks[i]
        for i in indices[0]
    ]

    context = "\n\n".join(relevant_chunks)

    prompt = f"""
    You are a helpful AI assistant that answers questions about my
    resume and projects.

    Answer the question using ONLY the context provided below.

    Rules:
- Do not invent information.
- Do not use outside knowledge.
- If the answer is not present in the context, say:
  "I couldn't find the answer in the provided documents."
- Give a clear and concise answer.
- If there are multiple relevant items, list them.
- Do not use:
- #
- ##
- ###
- *
- **
- ```
- ---
- Markdown headings

CONTEXT:
{context}

QUESTION:
{question}
"""


    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt
    )

    return response.text

In [42]:
while True:

    question = input("\nYou: ")

    if question.lower() in ["exit", "quit", "bye"]:
        break

    answer = ask_question(question)

    print("\nAI:", answer)


AI: Based on the provided documents, you have completed 5 projects:

1. EduVoice - AI Lecture Learning Assistant
2. AI Resume Parser
3. Credit Card Fraud Detection System (AI Fraud Detection System)
4. Employee Attrition Prediction
5. Movie Recommendation System

AI: I couldn't find the answer in the provided documents.

AI: whyshaheer143@gmail.com

AI: I couldn't find the answer in the provided documents.


ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'EmbedContentRequest.content contains an empty Part.', 'status': 'INVALID_ARGUMENT'}}